# 2 · Refusing a reply, and saying why

Companion to the tutorial *From Hand-Crafted to LLM-Based Variation Operators in Metaheuristics*. It runs offline: the model is a fixed pool of completions, so every number below comes out the same on your machine.

A model asked for a tour will sometimes return prose, sometimes a list with a
city missing, sometimes a perfectly formed answer to a different question. The
validator decides which of those become candidates.

The part people get wrong is not the refusing. It is what the refusal says. A
validator that answers *invalid* tells the model nothing it can act on, and then
the retry is a lottery. This notebook walks the three layers, and then shows what
one retry buys when the message is specific.

In [1]:
import _bootstrap
import tsp_transient as tsp
from llm import load_pool

## Layer 1 · is there an answer in here at all

Cheapest check first. No envelope, no candidate, and nothing downstream runs.

In [2]:
try:
    tsp.parse('Sure! A good tour would be 0 -> 1 -> 2 -> 3 -> 4. Let me know if…')
except ValueError as err:
    print(err)

schema error: no CANDIDATE envelope


## Layer 2 · does the payload parse

The envelope is there. What it carries is not a list of integers.

In [3]:
reply = '''CANDIDATE
id: t
representation: permutation
payload:
the usual order, starting from the depot
END_CANDIDATE'''

try:
    tsp.parse(reply)
except ValueError as err:
    print(err)

syntax error: no payload list


## Layer 3 · is it legal in this problem

This one parses. It is a list of integers and it is still not a tour. Note what
the message names.

In [4]:
bad = tsp.parse(load_pool('tsp_pool.txt')[0])
print('parsed payload:', bad)

try:
    tsp.feasible(bad)
except ValueError as err:
    print(err)

parsed payload: [0, 1, 4, 4, 2]
feasibility: city [4] duplicated, city [3] missing


In [5]:
import viz

coords = [tsp.COORDS[i] for i in range(len(tsp.COORDS))]
viz.tours(coords, [([0, 2, 3, 4, 1], 'the incumbent', 'a legal tour'),
                   (bad, 'the first sample', 'city 4 twice, city 3 never')])

'<svg xmlns="http://www.w3.org/2000/svg" width="400" height="226" viewBox="0 0 400 226" font-family="ui-sans-serif,-apple-system,Segoe UI,Roboto,sans-serif"><rect width="400" height="226" fill="#ffffff"/><g transform="translate(0,0)"><rect x="4" y="4" width="192" height="214" rx="9" fill="none" stroke="#d8dce3" stroke-width="1"/><text x="100.0" y="20.0" font-size="12" fill="#1f2430" text-anchor="middle" font-weight="600">the incumbent</text><polygon points="26.0,188.0 174.0,188.0 174.0,40.0 26.0,40.0 100.0,188.0" fill="#d9456b" fill-opacity="0.07" stroke="#d9456b" stroke-width="1.8" stroke-linejoin="round"/><circle cx="26.0" cy="188.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="26.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">0</text><text x="26.0" y="173.0" font-size="9" fill="#8a8f9a" text-anchor="middle" font-weight="normal">start</text><circle cx="174.0" cy="188.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="174.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">2</text><circle cx="174.0" cy="40.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="174.0" y="44.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">3</text><circle cx="26.0" cy="40.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="26.0" y="44.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">4</text><circle cx="100.0" cy="188.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="100.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">1</text><text x="100.0" y="210.0" font-size="11" fill="#8a8f9a" text-anchor="middle" font-weight="normal">a legal tour</text></g><g transform="translate(200,0)"><rect x="4" y="4" width="192" height="214" rx="9" fill="none" stroke="#d8dce3" stroke-width="1"/><text x="100.0" y="20.0" font-size="12" fill="#1f2430" text-anchor="middle" font-weight="600">the first sample</text><polygon points="26.0,188.0 100.0,188.0 26.0,40.0 26.0,40.0 174.0,188.0" fill="#1f9d78" fill-opacity="0.07" stroke="#1f9d78" stroke-width="1.8" stroke-linejoin="round"/><circle cx="26.0" cy="188.0" r="10" fill="#ffffff" stroke="#1f9d78" stroke-width="1.6"/><text x="26.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">0</text><text x="26.0" y="173.0" font-size="9" fill="#8a8f9a" text-anchor="middle" font-weight="normal">start</text><circle cx="100.0" cy="188.0" r="10" fill="#ffffff" stroke="#1f9d78" stroke-width="1.6"/><text x="100.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">1</text><circle cx="26.0" cy="40.0" r="10" fill="#ffffff" stroke="#1f9d78" stroke-width="1.6"/><text x="26.0" y="44.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">4</text><circle cx="26.0" cy="40.0" r="10" fill="#ffffff" stroke="#1f9d78" stroke-width="1.6"/><text x="26.0" y="44.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">4</text><circle cx="174.0" cy="188.0" r="10" fill="#ffffff" stroke="#1f9d78" stroke-width="1.6"/><text x="174.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">2</text><text x="100.0" y="210.0" font-size="11" fill="#8a8f9a" text-anchor="middle" font-weight="normal">city 4 twice, city 3 never</text></g></svg>'

The picture is the argument for layer 3. Both drawings are closed paths. Only one
of them visits every city once, and no amount of parsing would have caught the
difference: the payload was well formed, it was just wrong.

## The three layers are ordered by cost

| layer | what it costs | what it catches |
|---|---|---|
| envelope and schema | a string search | replies that are not answers |
| payload syntax | a parse | answers in the wrong shape |
| feasibility | domain work, sometimes running code | answers that are the wrong answer |

Running them in that order means a reply that is pure prose never reaches the
part that executes anything. With code as the payload, layer 3 means running what
the model wrote, and then the order stops being a nicety.

## What the message is for

The diagnostic is not a log line. It goes into the next prompt, verbatim. Here is
the repair prompt the loop builds from the failure above:

In [6]:
spec = tsp.Spec()
prompt = spec.render([0, 2, 3, 4, 1], tsp.tour_len([0, 2, 3, 4, 1]), [])

try:
    tsp.feasible(bad)
except ValueError as err:
    print(spec.repair_prompt(prompt, '', str(err)))

[context] TSP toy instance; minimize Euclidean closed-tour length.
[conditioning] R = permutation of [0, 1, 2, 3, 4] starting at 0; incumbent [0, 2, 3, 4, 1], score 9.236.
[instruction] Emit one lower-length tour if possible.
[format] Use the CANDIDATE envelope.
[repair] previous payload failed validation: feasibility: city [4] duplicated, city [3] missing. Emit a corrected CANDIDATE.


## One retry, measured

Same pool, same model, same everything. The only change is the retry budget.

In [7]:
from search import build_and_validate
from llm import MockLLM

def run(retries):
    llm = MockLLM(load_pool('tsp_pool.txt'))
    best, score, log = build_and_validate(
        llm, tsp.tour_len, lambda c, sc, cur, scur: sc < scur,
        [0, 2, 3, 4, 1], tsp.Spec(), budget=1, retries=retries, minimize=True)
    return best, score, log

for retries in (0, 1):
    best, score, log = run(retries)
    print(f'retries={retries}: {log[0][1]:8s}  best {best}  length {score:.3f}')

retries=0: invalid   best [0, 2, 3, 4, 1]  length 9.236
retries=1: accepted  best [0, 1, 2, 3, 4]  length 8.000


With no retry the step is spent: the reply was infeasible and nothing replaces it.
With one retry the same first reply is refused, the diagnostic goes back, and the
next completion is the optimum. One knob, and the step goes from wasted to solved.

That is the case for bounded repair. The bound matters too: a model that cannot
fix its answer in one or two attempts rarely fixes it in ten, and each attempt is
a paid call. The worst case is `budget × (retries + 1)` calls, which is the number
to put in a cost table.

## Try it

1. Write a reply whose payload is `[0, 1, 2, 3, 4, 5]` and see which layer catches
   it. Is the message specific enough for a model to act on?
2. Weaken `feasible` so it only checks the length of the list, then re-run the
   demo. The search still runs, and the answers get quietly worse. A weak
   validator does not fail loudly; it just stops protecting you.
3. Raise `retries` to 3 with a pool of only invalid replies and count the calls.

---

Next: [3 · The model proposes the solution](03_tsp_transient.ipynb)